# Probabilistic ML Thinking (Naive Bayes) — AI Lab Instructor Notebook

*Classical ML · Medium*

Build probabilistic intuition for generative modeling and implement Naive Bayes from scratch.

**Outcome.** Students can explain generative vs discriminative modeling, conditional independence assumptions, and implement Naive Bayes for text classification (log-space, Laplace smoothing).

7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import re
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Naive Bayes — Student Lab

Implement Multinomial Naive Bayes from scratch (log-space + Laplace smoothing).

## Section 0 — Toy dataset
We use a tiny spam/ham dataset so you can verify logic easily.

In [ ]:
import re
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

docs = [
    ('spam', 'win money now'),
    ('spam', 'free money win'),
    ('spam', 'claim your free prize'),
    ('ham',  'meeting schedule tomorrow'),
    ('ham',  'let us schedule a meeting'),
    ('ham',  'project update tomorrow'),
]
labels = np.array([1 if y=='spam' else 0 for y,_ in docs])
texts = [t for _,t in docs]

**Why this works.** # Naive Bayes — Instructor (Solutions + Rationale)

**Naive Bayes Model**

Probabilist model based on Baye's theorem

**Naive Bayes Assumption**

Events are independent

## Task 2: Tokenize
*Section: Tokenization + Vocabulary*

## Section 1 — Tokenization + Vocabulary

### Task 1.1: Tokenize


**FAANG gotcha:** punctuation/case can break counts.

**Hints**
- lowercase + split on non-letters

In [ ]:
def tokenize(text):
    tokens = re.split(r'[^a-z]+', text.lower())
    return [t for t in tokens if t]

vocab = {}
for t in texts:
    for w in tokenize(t):
        if w not in vocab:
            vocab[w] = len(vocab)
check('vocab_nonempty', len(vocab) > 0)
# print(vocab)
inv_vocab = {i:w for w,i in vocab.items()}
# print(inv_vocab)

## Task 3: Build vocabulary
*Section: Tokenization + Vocabulary*

### Task 1.2: Build vocabulary


**Checkpoint:** What happens if a word appears only in test?

In [ ]:
def tokenize(text):
    tokens = re.split(r'[^a-z]+', text.lower())
    return [t for t in tokens if t]

vocab = {}
for t in texts:
    for w in tokenize(t):
        if w not in vocab:
            vocab[w] = len(vocab)
check('vocab_nonempty', len(vocab) > 0)
# print(vocab)
inv_vocab = {i:w for w,i in vocab.items()}
# print(inv_vocab)

In [ ]:
# Checks
check('vocab_nonempty', len(vocab) > 0)

## Task 4: Convert docs to count matrix X (n_docs, |V|)
*Section: Vectorization (Count vectors)*

## Section 2 — Vectorization (Count vectors)

### Task 2.1: Convert docs to count matrix X (n_docs, |V|)

**Hints**
- loop over tokens inside a doc is ok; avoid loops over vocab per doc

In [ ]:
def vectorize(texts, vocab):
    X = np.zeros((len(texts), len(vocab)), dtype=np.int64)
    # print(X)
    for i, text in enumerate(texts):
        for w in tokenize(text):
            if w in vocab:
                X[i, vocab[w]] += 1
    return X

X = vectorize(texts, vocab)
print(X)
check('shape', X.shape[0] == len(texts) and X.shape[1] == len(vocab))

In [ ]:
# Checks
check('shape', X.shape[0] == len(texts) and X.shape[1] == len(vocab))

## Task 5: Fit with Laplace smoothing
*Section: Train Multinomial Naive Bayes*

## Section 3 — Train Multinomial Naive Bayes

### Task 3.1: Fit with Laplace smoothing

Compute:
- class priors P(y)
- word likelihoods P(word|y) with Laplace smoothing alpha

- count words per class: sum rows where y==c
- add alpha to each word count

**Checkpoint:** Why smoothing is required?

### Figure: Learned Class Priors

![Class Priors](/images/w3-d5-nb-class-priors.png)

**Hints**
- count words per class: sum rows where y==c
add alpha to each word count

In [ ]:
def fit_nb(X, y, alpha=1.0):
    y = y.astype(int)
    V = X.shape[1]
    log_priors = np.zeros(2)
    log_lik = np.zeros((2, V))

    for c in (0, 1):
        Xc = X[y == c]
        # print(Xc)
        log_priors[c] = np.log(Xc.shape[0] / X.shape[0])
        word_counts = Xc.sum(axis=0)
        print(word_counts)
        smoothed = word_counts + alpha
        denom = smoothed.sum()
        log_lik[c] = np.log(smoothed / denom)

    return log_priors, log_lik

log_priors, log_lik = fit_nb(X, labels, alpha=1.0)
check('priors_shape', log_priors.shape == (2,))
check('lik_shape', log_lik.shape == (2, X.shape[1]))

# Rationale: Laplace smoothing prevents zero probability for unseen words.

In [ ]:
# Checks
check('priors_shape', log_priors.shape == (2,))
check('lik_shape', log_lik.shape[0] == 2 and log_lik.shape[1] == X.shape[1])

**Why this works.** Laplace smoothing prevents zero probability for unseen words.

Multinomial Naive Bayes (MNB) is a probabilistic classifier used mainly for text data, where inputs are count vectors (bag-of-words)

```
Laplace (add-one) smoothing.

The correct formula

For Multinomial Naive Bayes:

𝑝 = (count(word in class) + 1)/(total words in that class + ∣𝑉∣)

```

Why + |V|?

Because when you add +1 to every word count, you’ve added |V| extra pseudo-counts in total.

## Task 6: Predict in log-space
*Section: Train Multinomial Naive Bayes*

### Task 3.2: Predict in log-space

For each doc:
log P(y=c|x) â log P(y=c) + sum_j x_j * log P(word_j|c)

**Interview Angle:** Why use logs?

### Figure: Confusion Matrix

![NB Confusion Matrix](/images/w3-d5-nb-confusion-matrix.png)

In [ ]:
def predict_nb(X, log_priors, log_lik):
    # scores: (n, 2)
    scores0 = log_priors[0] + X @ log_lik[0]
    scores1 = log_priors[1] + X @ log_lik[1]
    scores = np.vstack([scores0, scores1]).T
    # print(scores)
    return np.argmax(scores, axis=1)

pred = predict_nb(X, log_priors, log_lik)
acc = float(np.mean(pred == labels))
print('train_acc', acc)
check('acc_reasonable', acc >= 0.8)

# Rationale: log-space turns product into sum, avoiding underflow.

In [ ]:
# Checks
check('acc_reasonable', acc >= 0.8)

**Why this works.** log-space turns product into sum, avoiding underflow.

## Task 7: Most influential tokens per class
*Section: Error Analysis*

## Section 4 — Error Analysis

### Task 4.1: Most influential tokens per class
Show top tokens by log-likelihood ratio between classes.

### Figure: Top Spam Tokens

![Top Tokens](/images/w3-d5-nb-top-tokens.png)

**Hints**
- sort (log_lik[spam] - log_lik[ham])

In [ ]:
score = log_lik[1] - log_lik[0]
top = np.argsort(-score)[:5]
print('top spam tokens:', [inv_vocab[i] for i in top])

# Rationale: log-likelihood ratio shows which tokens push prediction toward spam.

**Why this works.** log-likelihood ratio shows which tokens push prediction toward spam.